# AI Evaluation

- AI Evaluation is the most critical part of getting an agent pilot into production. It gives a quantitative view into agent performance and enables a clear understanding of model performance that builds the trust with stakeholders and end users necessary to drive towards production.

- We will use built-in judges, custom guideline based judges, and a fully custom prompt based judge to evaluate performance

- This gives a full view into how the agent is performing, and the trace-level observational detail allows us to drill into specific examples to determine quality

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph
dbutils.library.restartPython()

In [ ]:
import json
from pathlib import Path
import mlflow

# Load configuration from setup notebook
CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
# Extract configuration variables
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
LABEL_SCHEMA_NAME = CONFIG["evaluation"]["label_schema_name"]
LABELING_SESSION_NAME = CONFIG["evaluation"]["labeling_session_name"]
ASSIGNED_USERS = CONFIG["evaluation"]["assigned_users"]
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
EVAL_GENERATION_MODEL = JUDGE_MODEL.replace("databricks:/", "")
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

## Create Fresh Evaluation Dataset

Build a fresh full evaluation set for each run instead of reusing a partially populated MLflow GenAI dataset.

Player comparison prompts are intentionally excluded from this evaluation set.

In [ ]:
from mlflow.genai.datasets import get_dataset


def _extract_question_from_record(record: dict) -> str | None:
    """Best-effort extraction of the user question from a dataset record."""
    if not isinstance(record, dict):
        return None

    inputs = record.get("inputs")
    if isinstance(inputs, dict):
        input_messages = inputs.get("input")
        if isinstance(input_messages, list) and input_messages:
            first_msg = input_messages[0]
            if isinstance(first_msg, dict):
                content = first_msg.get("content")
                return content if isinstance(content, str) else None

    # Fallback shape seen in some trace-derived records
    request = record.get("request")
    if isinstance(request, dict):
        input_messages = request.get("input")
        if isinstance(input_messages, list) and input_messages:
            first_msg = input_messages[0]
            if isinstance(first_msg, dict):
                content = first_msg.get("content")
                return content if isinstance(content, str) else None

    return None


def _try_load_existing_eval_records(dataset_name: str) -> list[dict]:
    """Load existing dataset records directly if dataset exists."""
    try:
        eval_ds = get_dataset(name=dataset_name)
        eval_df = eval_ds.to_df()
    except Exception:
        return []

    eval_data = []
    for _, row in eval_df.iterrows():
        inputs = row.get("inputs")
        if inputs is None:
            continue

        if isinstance(inputs, str):
            inputs = json.loads(inputs)

        if isinstance(inputs, dict) and "request" in inputs and isinstance(inputs["request"], dict):
            # Trace-derived Responses schema: {"request": {"input": [...]}}
            request_obj = inputs["request"]
            msg_input = request_obj.get("input", [])
            if not isinstance(msg_input, list):
                msg_input = [msg_input]
            entry = {"inputs": {"input": msg_input}}
        elif isinstance(inputs, dict) and "input" in inputs:
            # Already in evaluate()-friendly shape
            msg_input = inputs["input"]
            if not isinstance(msg_input, list):
                msg_input = [msg_input]
            entry = {"inputs": {"input": msg_input}}
        else:
            msg_input = inputs if isinstance(inputs, list) else [inputs]
            entry = {"inputs": {"input": msg_input}}

        expectations = row.get("expectations")
        if expectations is not None:
            if isinstance(expectations, str):
                expectations = json.loads(expectations)
            if expectations:
                entry["expectations"] = expectations

        eval_data.append(entry)

    return eval_data


FORCE_REGENERATE_EVAL_DATASET = True
STATIC_EVAL_QUESTIONS = [
    "How did Freddie Freeman perform against Kyle Freeland in 2025, and what should his plate approach be?",
    "How does Blake Snell usually attack Mookie Betts in 0-2 counts against right-handed batters in 2025?",
    "What pitch types does Yu Darvish throw, and which ones should left-handed batters prepare for in 2025?",
    "For Aaron Judge against Framber Valdez in 2025, summarize the matchup history and recommend an approach.",
    "How does Gerrit Cole pitch with runners on first and second against right-handed batters in 2025?",
    "What are Zack Wheeler's tendencies in 3-2 counts against left-handed batters in 2025?",
    "List the LAD batters available in 2025 and recommend a lineup approach against Logan Webb.",
    "What pitch types does Logan Webb throw in 2025? Keep the answer concise.",
    "How should Shohei Ohtani approach a matchup with Zac Gallen based on their 2025 matchup data?",
    "What are Corbin Burnes' pitch tendencies with a runner on second against left-handed batters in 2025?",
    "For Juan Soto against Spencer Strider in 2024, summarize the matchup and give a hitter recommendation.",
    "What are Max Fried's tendencies in 1-1 counts against right-handed batters in 2025?",
    "Which NYY batters should be prioritized against Kevin Gausman in 2025 based on available matchup tools?",
    "How does Luis Castillo pitch with runners on first and third against right-handed batters in 2025?",
    "What pitch types does Tarik Skubal throw in 2025, and what should hitters expect?",
    "For COL pitchers in 2025, what was the average release speed by pitch type? Keep the answer concise.",
    "What was the pitch distribution for LAD pitchers in 2025 by pitch type?",
    "Which pitchers had the highest average four-seam fastball velocity in 2025?",
    "Compare average release speed and spin rate by pitch type for NYY and BOS pitchers in 2025.",
    "What percentage of Blake Snell's 2025 pitches were breaking balls?",
    "What was the average launch angle for ATL batters in 2025?",
    "Which team had the highest average exit velocity in 2025?",
    "For SEA pitchers in 2025, show pitch type usage and average spin rate by pitch type.",
    "How did Gerrit Cole's average fastball velocity change from 2024 to 2025?",
    "What was the zone distribution for Logan Webb's pitches in 2025?",
    "Compare pitch usage between HOU and TEX pitchers in 2025.",
    "Which 2025 pitchers threw sliders at the highest rate?",
    "What was the average barrel rate for NYM batters in 2025?",
    "For PHI batters in 2025, what were the average launch speed and estimated wOBA by batter?",
    "Across the league in 2025, which pitch type had the highest average spin rate?",
]

# Reusing the dataset caused a prior run to evaluate only three stale records.
# Keep this default fresh so 04 produces a full trace set each time.
eval_dataset_records = [] if FORCE_REGENERATE_EVAL_DATASET else _try_load_existing_eval_records(DATASET_NAME)

if FORCE_REGENERATE_EVAL_DATASET:
    print(f"Using {len(STATIC_EVAL_QUESTIONS)} fresh static evaluation questions; existing dataset records are ignored.")
    raw_output = json.dumps(STATIC_EVAL_QUESTIONS)
elif eval_dataset_records:
    print(f"Loaded {len(eval_dataset_records)} evaluation records from dataset '{DATASET_NAME}'.")
elif not eval_dataset_records:
    print(f"No records found in dataset '{DATASET_NAME}'. Generating via FMAPI...")

    from openai import OpenAI
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    fmapi_client: OpenAI = w.serving_endpoints.get_open_ai_client()

    GENERATION_PROMPT = """You are an expert at generating evaluation examples for a baseball hitting analysis AI assistant.

The assistant helps batters prepare for matchups against specific pitchers. It has two kinds of tools:

## UC Functions (deterministic tools)
These answer specific, structured questions:
- lookup_player_by_name: Resolve player name to ID
- get_batter_pitcher_matchup: Historical pitches between specific batter-pitcher pairs
- get_pitcher_tendency_by_count: Pitch type/location by count (balls, strikes) and batter hand
- get_pitcher_tendency_with_runners: Same as above but filtered by base runner situation
- pitcher_arsenal_lookup: Get all pitch types a pitcher throws
- recommend_batter_matchups_by_team: Best lineup matchups vs a pitcher
- get_team_batters: Full batter roster for a team

## Genie Space (SQL-based analytics)
The assistant also has a Genie Space that can query underlying tables directly for questions the UC functions cannot answer:
- statcast_pitches: All pitch-level data
- dim_pitchers, dim_batters, dim_players: Player dimension tables
- dim_pitcher_arsenal: Pitcher arsenal summary
- dim_batter_team_year, dim_pitcher_team_year: Team-season rosters

Generate exactly 30 diverse evaluation examples as a JSON array. Each example should be a single string (the user question).

IMPORTANT: Include a balanced mix of BOTH types:

### ~15 UC Function questions (answerable by the tools above):
- Specific batter vs pitcher matchup analysis
- Pitcher tendencies by count and batter hand (with specific count numbers like 0-2, 1-1, 3-2)
- Pitcher tendencies with runners on specific bases
- Pitcher arsenal lookups
- Lineup construction against a specific pitcher
- Team roster queries
- Do NOT generate player comparison or comp questions

### ~15 Genie-only questions (require SQL analytics, NOT answerable by UC functions):
- Pitch distribution (% of each pitch type) for a specific team in a season
- Average velocity or spin rate across a team's pitching staff
- Which pitchers throw the hardest fastball in the league?
- What is the average launch angle for batters on a specific team?
- Compare pitch usage between two teams
- What percentage of pitches are breaking balls for a specific pitcher?
- Team-level batting statistics (avg exit velocity, barrel rate)
- Historical trends: how has a pitcher's velocity changed across seasons?
- Distribution of pitch locations (zone analysis) for a pitcher
- League-wide statistics and comparisons

Use real MLB player names and teams. Use the 3-letter team abbreviations: TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

Return ONLY a valid JSON array of strings, no other text.

Only ask for data from 2024 & 2025 seasons."""

    response = fmapi_client.chat.completions.create(
        model=EVAL_GENERATION_MODEL,
        messages=[{"role": "user", "content": GENERATION_PROMPT}],
        temperature=0.8,
        max_tokens=4000,
    )

    raw_content = response.choices[0].message.content
    if isinstance(raw_content, list):
        raw_output = "".join(
            part.get("text") or part.get("content") or ""
            for part in raw_content
            if isinstance(part, dict)
        )
    else:
        raw_output = str(raw_content)

    print(f"Generated evaluation examples with model: {EVAL_GENERATION_MODEL}")
    print("Raw FMAPI output (first 200 chars):")
    print(raw_output[:200])

In [ ]:
import re

# If we loaded existing records, just preview them.
# Otherwise parse FMAPI output and create records.
if eval_dataset_records:
    print(f"Reusing {len(eval_dataset_records)} evaluation records from existing dataset '{DATASET_NAME}'.\n")
    for i, rec in enumerate(eval_dataset_records):
        question = rec["inputs"]["input"][0]["content"]
        print(f"  {i+1:2d}. {question}")
else:
    json_match = re.search(r'\[.*\]', raw_output, re.DOTALL)
    if json_match:
        example_questions = json.loads(json_match.group())
    else:
        example_questions = json.loads(raw_output)

    banned_terms = ("similar", "player comp", "batter comp", "hitter comp")
    filtered_questions = [
        q for q in example_questions
        if not any(term in q.lower() for term in banned_terms)
    ]
    removed_count = len(example_questions) - len(filtered_questions)
    if removed_count:
        print(f"Removed {removed_count} player comparison evaluation questions")
    example_questions = filtered_questions
    if len(example_questions) != 30:
        raise ValueError(f"Expected 30 evaluation questions after filtering, got {len(example_questions)}")

    print(f"Generated {len(example_questions)} evaluation questions\n")

    for i, q in enumerate(example_questions):
        print(f"  {i+1:2d}. {q}")

    eval_dataset_records = [
        {
            "inputs": {
                "input": [
                    {"role": "user", "content": question}
                ]
            }
        }
        for question in example_questions
    ]

print(f"\nPrepared {len(eval_dataset_records)} evaluation records")

## Define Judges / Scorers

We use three types of judges:
1. **RelevanceToQuery** - Built-in judge for response relevance
2. **Guidelines** - Custom guideline judge for baseball-specific language
3. **make_judge** - Fully custom judge for baseball analysis quality (1-5 Likert scale)

In [ ]:
import logging
logging.getLogger("mlflow.genai.judges.instructions_judge").setLevel(logging.ERROR)

from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
    Scorer,
)
from mlflow.entities import Feedback
import json as _json
import re as _re



class DataToolAvailabilityScorer(Scorer):
    """Scores whether the run had usable retrieved evidence separate from answer quality."""
    name: str = "data_tool_availability"

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        text = _json.dumps(outputs, default=str).lower()
        timeout_patterns = [
            "execution timed out",
            "timed out after",
            "read timed out",
            "provisioning resources",
            "daily limit",
            "endpoint_not_found",
            "sql_execution_exception",
        ]
        no_data_patterns = [
            "no relevant records",
            "no rows",
            "empty result",
            "don’t have any information",
            "don't have any information",
            "does not include information",
        ]
        usable_patterns = [
            '"function_call_output"',
            "function_call_output",
            '"rows":[[',
            "| pitch_",
            "pitch_count",
            "frequency_pct",
            "avg_release_speed",
        ]

        if any(p in text for p in timeout_patterns):
            return Feedback(name=self.name, value=0.0, rationale="Tool or model execution failed before usable evidence was available.")
        if any(p in text for p in usable_patterns):
            return Feedback(name=self.name, value=1.0, rationale="The trace includes usable tool or Genie evidence.")
        if any(p in text for p in no_data_patterns):
            return Feedback(name=self.name, value=0.25, rationale="The run appears to have completed but found limited or no supporting data.")
        return Feedback(name=self.name, value=0.5, rationale="Data availability is unclear from the model output.")


data_tool_availability_scorer = DataToolAvailabilityScorer()

# Guideline judge for baseball-specific language
baseball_language = "The response must use language that is appropriate for professional baseball players and coaches. It should reference baseball-specific terminology accurately."
baseball_language_judge = Guidelines(name="baseball_language", guidelines=baseball_language)

# Custom judge for baseball analysis quality (1-5 scale)
baseball_analysis_judge = make_judge(
    name=ALIGNED_JUDGE_NAME,
    instructions=(
        "Evaluate if the response in {{ outputs }} appropriately analyzes the available data and provides an actionable recommendation "
        "to the question in {{ inputs }}. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. "
        "Your grading criteria should be: "
        " 1: Completely unacceptable. Incorrect data interpretation or no recommendations"
        " 2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage"
        " 3: Somewhat acceptable. Relevant feedback provided with some strategic advantage"
        " 4: Mostly acceptable. Relevant feedback provided with strong strategic advantage"
        " 5: Completely acceptable. Relevant feedback provided with excellent strategic advantage"
    ),
    feedback_value_type=float,
    model=JUDGE_MODEL,  # Model used to evaluate (from config)
)

scorers = [RelevanceToQuery(), baseball_analysis_judge, baseball_language_judge, data_tool_availability_scorer]

# Register judge to experiment (skip if already registered)
try:
    registered_base_judge = baseball_analysis_judge.register(experiment_id=EXPERIMENT_ID)
    print(f"Registered base judge: {registered_base_judge.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Base judge '{ALIGNED_JUDGE_NAME}' already registered (OK)")
    else:
        print(f"Warning registering judge: {e}")

## Run Evaluation

In [ ]:
import os

from mlflow.deployments import get_deploy_client
from mlflow.genai import evaluate

os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "300"
os.environ["MLFLOW_HTTP_REQUEST_MAX_RETRIES"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"

SERVING_ENDPOINT_NAME = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
deploy_client = get_deploy_client("databricks")


def _normalize_messages(input_payload):
    """Normalize evaluate() inputs into ResponsesAgentRequest input messages."""
    if isinstance(input_payload, dict) and "request" in input_payload and isinstance(input_payload["request"], dict):
        req_input = input_payload["request"].get("input", [])
        return req_input if isinstance(req_input, list) else [req_input]
    if isinstance(input_payload, dict) and "input" in input_payload:
        req_input = input_payload["input"]
        return req_input if isinstance(req_input, list) else [req_input]
    if isinstance(input_payload, list):
        return input_payload
    return [{"role": "user", "content": str(input_payload)}]


def _as_dict(resp):
    if isinstance(resp, dict):
        return resp
    to_dict_fn = getattr(resp, "as_dict", None) or getattr(resp, "to_dict", None)
    return to_dict_fn() if callable(to_dict_fn) else {"raw": str(resp)}


def _eval_predict_fn(input):
    messages = _normalize_messages(input)
    return _as_dict(deploy_client.predict(
        endpoint=SERVING_ENDPOINT_NAME,
        inputs={"input": messages},
    ))


print(f"Evaluating deployed endpoint: {SERVING_ENDPOINT_NAME}")
results = evaluate(
    data=eval_dataset_records,
    predict_fn=_eval_predict_fn,
    scorers=scorers,
)


In [ ]:
import numpy as np


def _extract_all_assessment_scores(run_id):
    """Collect numeric scores by assessment name from trace assessments."""
    traces_df = mlflow.search_traces(run_id=run_id)
    by_name = {}

    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            name = a.get("assessment_name") or "(unknown_assessment)"
            val = a.get("feedback", {}).get("value")
            score = None
            if val == "yes":
                score = 1.0
            elif val == "no":
                score = 0.0
            elif isinstance(val, (int, float)):
                score = float(val)

            if score is not None:
                by_name.setdefault(name, []).append(score)

    return by_name


print("=" * 60)
print("EVALUATION RESULTS SUMMARY")
print("=" * 60)

print("\n1) Raw metrics object:")
print(results.metrics if getattr(results, "metrics", None) else "(empty)")

print("\n2) Score summary (metrics + trace assessments):")
if isinstance(getattr(results, "metrics", None), dict) and results.metrics:
    for k, v in results.metrics.items():
        if isinstance(v, (int, float)):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
else:
    print("(No numeric metrics in results.metrics)")

assessment_scores = _extract_all_assessment_scores(results.run_id)
if assessment_scores:
    print("\nTrace assessment aggregates:")
    for name, scores in sorted(assessment_scores.items()):
        print(f"- {name}: mean={np.mean(scores):.4f} +/- {np.std(scores):.4f} (n={len(scores)})")
else:
    print("\nNo numeric trace assessments found.")

# Explicitly call out custom judge presence
custom_judge_present = ALIGNED_JUDGE_NAME in assessment_scores
print(f"\nCustom judge '{ALIGNED_JUDGE_NAME}' present: {custom_judge_present}")
if custom_judge_present:
    cj_scores = assessment_scores[ALIGNED_JUDGE_NAME]
    print(f"Custom judge mean: {np.mean(cj_scores):.4f} +/- {np.std(cj_scores):.4f} (n={len(cj_scores)})")

print("\n3) Run and table info:")
print(f"Run ID: {results.run_id}")
if hasattr(results, "tables") and results.tables:
    for tname, tdf in results.tables.items():
        print(f"- Table '{tname}': {len(tdf)} rows")

## Tag Successful Traces

Tag all traces with OK status as `eval: complete` for downstream alignment and optimization.

In [ ]:
# Grab the trace_id for all observations that have state of OK and apply the tag of eval: complete
ok_trace_ids = results.result_df.loc[results.result_df["state"] == "OK", "trace_id"]
print(f'Number of traces with OK status: {len(ok_trace_ids)}')

for trace_id in ok_trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="complete")

## Create GenAI Dataset from Traces

Collect the evaluated traces into an MLflow GenAI dataset for use in labeling sessions and judge alignment.

In [ ]:
from mlflow.genai.datasets import create_dataset, get_dataset

# Step 1: Create an empty evaluation dataset
try:
    eval_dataset = get_dataset(name=DATASET_NAME)
except Exception:
    eval_dataset = create_dataset(
        name=DATASET_NAME,
    )

print(f"Configured evaluation dataset: {eval_dataset.name}")

# Step 2: Grab only traces from the current evaluation run and add them to the dataset.
# This avoids merging stale eval traces from earlier broken endpoint runs.
print(f"\nSearching for current run traces with tag 'eval: complete' in run {results.run_id}...")
traces_with_tag = mlflow.search_traces(
    run_id=results.run_id,
    filter_string="tag.eval = 'complete'",
    return_type="pandas"
)

print(f"Found {len(traces_with_tag)} traces with tag 'eval: complete'")

# Convert dataset to align with inputs needed for merge_traces()
if 'inputs' not in traces_with_tag.columns and 'request' in traces_with_tag.columns:
    print("Renaming 'request' column to 'inputs'...")
    traces_with_tag = traces_with_tag.rename(columns={'request': 'inputs'})

if 'outputs' not in traces_with_tag.columns and 'response' in traces_with_tag.columns:
    print("Renaming 'response' column to 'outputs'...")
    traces_with_tag = traces_with_tag.rename(columns={'response': 'outputs'})

eval_dataset = eval_dataset.merge_records(traces_with_tag)

## Create Label Schema and Labeling Session (Review App)

Set up the Review App for domain experts to provide human feedback on agent responses.
This enables:
- SME labeling of response quality (1-5 scale)
- Collection of human feedback for judge alignment (Pillar 6)
- Continuous improvement through human-in-the-loop evaluation

In [ ]:
from mlflow.genai import create_labeling_session, get_review_app
from mlflow.genai import label_schemas

# Step 1: Create label schemas for collecting feedback
# Create a custom schema that matches the baseball_analysis_judge criteria (1-5 scale)
baseball_analysis_schema = label_schemas.create_label_schema(
    name=LABEL_SCHEMA_NAME,
    type="feedback",
    title=LABEL_SCHEMA_NAME,
    input=label_schemas.InputNumeric(
        min_value=1.0,
        max_value=5.0,
    ),
    instruction=(
        "Evaluate if the response appropriately analyzes the available data and provides an actionable recommendation "
        "for the question. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. "
        "\n\n Your grading criteria should be: "
        "\n 1: Completely unacceptable. Incorrect data interpretation or no recommendations"
        "\n 2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage"
        "\n 3: Somewhat acceptable. Relevant feedback provided with some strategic advantage"
        "\n 4: Mostly acceptable. Relevant feedback provided with strong strategic advantage"
        "\n 5: Completely acceptable. Relevant feedback provided with excellent strategic advantage"
    ),
    enable_comment=True,  # Allow additional comments/feedback
    overwrite=True,
)

# Step 2: Set up the Review App with the deployed agent
AGENT_NAME = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
MODEL_NAME_SHORT = UC_MODEL_NAME.split('.')[-1]
review_app = get_review_app(experiment_id=EXPERIMENT_ID)

# Add the agent to the review app
review_app = review_app.add_agent(
    agent_name=MODEL_NAME_SHORT,
    model_serving_endpoint=AGENT_NAME,
    overwrite=True,
)

In [ ]:
# Step 3: Create the labeling session & add the traces for evaluation

labeling_session = create_labeling_session(
    name=f'{LABELING_SESSION_NAME}_sme',
    assigned_users=ASSIGNED_USERS,
    label_schemas=[LABEL_SCHEMA_NAME],  # Required: define what feedback to collect
)
# Add the dataset to the labeling session
labeling_session = labeling_session.add_dataset(
    dataset_name=DATASET_NAME
)
print(f"Created labeling session: {labeling_session.name}")
print(f"Labeling session ID: {labeling_session.labeling_session_id}")
print(f"Assigned users: {labeling_session.assigned_users}")
print(f"Labeling session URL: {labeling_session.url}")

## Next Steps

Once the labeling sessions are completed by SMEs, proceed to:
- **05-JudgeAlignment.ipynb** - Align the judge with SME feedback using MemAlign
- **06-PromptOptimization.ipynb** - Optimize the agent prompt using the aligned judge
- **07-AgentSkillsGeneration.ipynb** - Generate agent skills using the optimized prompt
- **08_create_agent_with_skills.ipynb** - Build the skills-enhanced agent
- **09-Evaluation.ipynb** - Compare both agents on a held-out dataset